# Save the data and evaluation rules

## Objective

Turn a small table into a saved study with clear training and testing dates.

Start here for the Python API. This notebook is independent of notebook 1.

We create **84 rows: two products on every one of 42 dates**. There are no missing observations or exclusions. Real projects must decide the starting population, date range and exclusions before registration.

A **series** is one product's history. A **horizon** is the number of future periods. Here it is seven days.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


We also turn on progress messages, so we can see which step is running.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

from autoforecast import configure_logging

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

# Show the current step while the library is working.
configure_logging()


### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


Turning logging on does not run a forecast. Later library calls will show their steps. A stage count tells us how many steps have finished, not how much time is left.


## Step 2: Start the data engine

We will start a small local Spark session with two workers. Spark reads and checks the study data. UTC keeps date and time handling consistent.


In [ ]:
# Import the tools used in this step.
from pyspark.sql import SparkSession

# Use two local workers for this small example.
spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### Understand the output

Spark may print startup messages. The cell should finish without an exception. No forecasts have been made yet.


Create an ordinary Parquet table. Replace this cell with your own data loading later.


## Step 3: Create a small table

We will make one row per product and day, check that there are 84 rows, and save them as Parquet.


In [ ]:
# Import the tools used in this step.
import datetime as dt

import pyarrow as pa
import pyarrow.parquet as pq

# Make one observation for each product on each date.
rows = [
    {
        "product": product,
        "date": dt.date(2024, 1, 1) + dt.timedelta(days=day),
        "units": float(3 + day % 7 + offset),
    }
    for product, offset in (("A", 0), ("B", 2))
    for day in range(42)
]
assert len(rows) == 84
# Save the observations in a format Spark can read.
pq.write_table(pa.Table.from_pylist(rows), work / "input.parquet")

### Understand the output

No output is expected. The assertion checked that we made 84 observations. `input.parquet` now contains that table. If the assertion fails after you edit the example, check the number of products and dates.


The validation forecast starts after 28 January. The final test starts after 4 February. The final test is kept separate from choosing model settings. This example has no gaps. Missing training observations are left for your model to handle. Missing validation or test observations stop registration. Zero means a recorded zero.


## Step 4: Write the study settings

We will describe the column names, daily frequency, and training and forecast dates. Read the comments beside each section before changing these settings.


In [ ]:
# These settings describe the data and keep the final test separate.
study_yaml = """schema_version: 1
# Map the input columns to the fields AutoForecast expects.
dataset:
  schema_version: 1
  dataset:
    id: example
    panel:
      series_col: product
      time_col: date
      target_col: units
      frequency: D
      season_length: 7
      target_estimand: observed_nonnegative
# Choose the training dates and keep validation and final testing separate.
recipe:
  schema_version: 2
  source_keys: [product]
  frequency: D
  training_start: "2024-01-01"
  validation_origins: ["2024-01-28"]
  validation_horizon: 7
  final_train_cutoff: "2024-02-04"
  final_test_horizon: 7
  history_policy: fixed
  new_series_policy: error
  target_unit: units
  feature_availability: {}
  metric: mae
  # Keep missing training values missing; the model must handle them.
  history_missing: preserve
  # Refuse missing values needed to score validation or testing.
  actual_missing: error
# Read the small Parquet file created above.
source:
  kind: parquet
  path: input.parquet
# Keep saved studies and runs in a folder beside the configuration.
storage:
  root: saved-studies
"""
# Save the updated settings for the registry to read.
_ = (work / "study.yaml").write_text(study_yaml)

### Understand the output

No output is expected. `study.yaml` now contains the settings. Writing the file alone does not validate the data; the next step does that.


## Step 5: Register the study

We will load those settings, check the data, and save the study ID for the next notebook. This step does not train a model.


In [ ]:
# Import the tools used in this step.
from autoforecast import register_study
from autoforecast.study_registration import StudyRegistry

# Open the registry described by our saved study settings.
registry = StudyRegistry.from_config(work / "study.yaml", spark=spark)

# Check the data and save an ID for this exact study.
study_id = register_study(work / "study.yaml", registry=registry)
_ = (work / "study-id.txt").write_text(study_id)
print(study_id)

### Understand the output

You should see a long study ID. It identifies this particular data and rule set, not a model or a score. The same ID is saved in `study-id.txt`. If registration fails, fix the reported data or settings problem before training.


At INFO level, stage messages and a progress panel also show the work as it happens. The study ID described above is the result to keep. A completed-stage percentage counts steps; it does not predict how much time remains.


Registration checks and saves the data. It does not train a model. Repeating registration with the same data and rules reuses the study ID. Changed data or rules create another ID. Continue with notebook 3.


## Step 6: Close this Spark session

We are finished with Spark in this notebook. Closing it frees resources before the next lesson.


In [ ]:
# Release Spark resources before moving to another notebook.
spark.stop()

### Understand the output

No output is expected. Spark is now closed. Start a fresh kernel when the next notebook asks you to.
